In [82]:
import polars as pl
import pandas as pd
from unidecode import unidecode

In [83]:
goodreads_book_df = pl.read_csv('datalake/clean/cleaned_books_data.csv').rename({
    'isbn13':'id', 'isbn':'ISBN_10','crawl_source':'crawl_source_url'
})

# Clean the string
goodreads_book_df = goodreads_book_df.with_columns(
    [
        pl.col(col).str.strip_chars("'").str.strip_chars('"').alias(col)
        for col in goodreads_book_df.columns if goodreads_book_df.schema[col] == pl.Utf8
    ]
)

# Add source column
goodreads_book_df = goodreads_book_df.with_columns(
    pl.lit('goodreads').alias('crawl_source')
)

goodreads_book_df

ISBN_10,id,title,image_url,description,author,num_pages,language,review_count,kindle_price,publish_date,crawl_source_url,average_rating,series,crawl_source
str,str,str,str,str,str,i64,str,i64,str,i64,str,f64,str,str
"""1662528396""","""9781662528392""","""Cruel Winter with You""","""https://images-na.ssl-images-a…","""For two former childhood frien…","""Ali Hazelwood""",73,"""English""",11514,"""0.99""",1731398400000,"""https://www.goodreads.com/book…",3.71,"""Under The Mistletoe Collection""","""goodreads"""
"""0664266703""","""9780664266707""","""The Purpose Gap: Empowering Co…","""https://images-na.ssl-images-a…","""In The Purpose Gap, Patrick Re…","""Patrick B. Reyes""",256,"""English""",12,"""9.99""",1615878000000,"""https://www.goodreads.com/book…",4.13,null,"""goodreads"""
"""166252823X""","""9781662528231""","""All by My Elf""","""https://images-na.ssl-images-a…","""Secret crushes, spicy Christma…","""Olivia Dade""",55,"""English""",3892,"""0.99""",1731398400000,"""https://www.goodreads.com/book…",2.61,"""Under The Mistletoe Collection""","""goodreads"""
"""1400341124""","""9781400341122""","""Mostly What God Does: Reflecti…","""https://images-na.ssl-images-a…","""Mostly what God does is love y…","""Savannah Guthrie""",302,"""English""",726,"""14.99""",1708416000000,"""https://www.goodreads.com/book…",4.34,null,"""goodreads"""
"""1662529376""","""9781662529375""","""Only Santas in the Building""","""https://images-na.ssl-images-a…","""It’s beginning to look a lot l…","""Alexis Daria""",65,"""English""",3047,"""0.99""",1731398400000,"""https://www.goodreads.com/book…",3.39,"""Under The Mistletoe Collection""","""goodreads"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1580625339""","""9781580625333""","""Everything Ghost Book""","""https://images-na.ssl-images-a…","""Ghosts and spirits have haunte…","""Jason R. Rich""",289,"""English""",2,null,996649200000,"""https://www.goodreads.com/book…",3.5,null,"""goodreads"""
"""0879517107""","""9780879517106""","""Everything and More""","""https://images-na.ssl-images-a…","""“Powered by a fast-moving plot…","""Geoff Nicholson""",249,"""English""",5,"""12.99""",928220400000,"""https://www.goodreads.com/book…",3.88,null,"""goodreads"""
"""158062684X""","""9781580626842""","""The Everything Kids' Nature Bo…","""https://images-na.ssl-images-a…","""The natural world holds secret…","""Kathiann M. Kowalski""",144,"""English""",2,"""6.99""",1014969600000,"""https://www.goodreads.com/book…",4.0,null,"""goodreads"""


In [84]:
# Further cleaning
# Change publish date to string date
goodreads_book_df = goodreads_book_df.with_columns(
    pl.col("publish_date")
    .cast(pl.Datetime("ms")) # Convert to Datetime
    .dt.strftime("%Y-%m-%d") # Format as "yyyy-mm-dd"
)

goodreads_book_df = goodreads_book_df.filter(pl.col("id") != "None")

In [85]:
print(goodreads_book_df.describe())
goodreads_book_df

shape: (9, 16)
┌────────────┬─────────┬───────────────┬───────────────┬───┬──────────────┬──────────────┬──────────────┬──────────────┐
│ statistic  ┆ ISBN_10 ┆ id            ┆ title         ┆ … ┆ crawl_source ┆ average_rati ┆ series       ┆ crawl_source │
│ ---        ┆ ---     ┆ ---           ┆ ---           ┆   ┆ _url         ┆ ng           ┆ ---          ┆ ---          │
│ str        ┆ str     ┆ str           ┆ str           ┆   ┆ ---          ┆ ---          ┆ str          ┆ str          │
│            ┆         ┆               ┆               ┆   ┆ str          ┆ f64          ┆              ┆              │
╞════════════╪═════════╪═══════════════╪═══════════════╪═══╪══════════════╪══════════════╪══════════════╪══════════════╡
│ count      ┆ 77963   ┆ 77963         ┆ 77963         ┆ … ┆ 77963        ┆ 77963.0      ┆ 10415        ┆ 77963        │
│ null_count ┆ 0       ┆ 0             ┆ 0             ┆ … ┆ 0            ┆ 0.0          ┆ 67548        ┆ 0            │
│ mean       ┆ nu

ISBN_10,id,title,image_url,description,author,num_pages,language,review_count,kindle_price,publish_date,crawl_source_url,average_rating,series,crawl_source
str,str,str,str,str,str,i64,str,i64,str,str,str,f64,str,str
"""1662528396""","""9781662528392""","""Cruel Winter with You""","""https://images-na.ssl-images-a…","""For two former childhood frien…","""Ali Hazelwood""",73,"""English""",11514,"""0.99""","""2024-11-12""","""https://www.goodreads.com/book…",3.71,"""Under The Mistletoe Collection""","""goodreads"""
"""0664266703""","""9780664266707""","""The Purpose Gap: Empowering Co…","""https://images-na.ssl-images-a…","""In The Purpose Gap, Patrick Re…","""Patrick B. Reyes""",256,"""English""",12,"""9.99""","""2021-03-16""","""https://www.goodreads.com/book…",4.13,null,"""goodreads"""
"""166252823X""","""9781662528231""","""All by My Elf""","""https://images-na.ssl-images-a…","""Secret crushes, spicy Christma…","""Olivia Dade""",55,"""English""",3892,"""0.99""","""2024-11-12""","""https://www.goodreads.com/book…",2.61,"""Under The Mistletoe Collection""","""goodreads"""
"""1400341124""","""9781400341122""","""Mostly What God Does: Reflecti…","""https://images-na.ssl-images-a…","""Mostly what God does is love y…","""Savannah Guthrie""",302,"""English""",726,"""14.99""","""2024-02-20""","""https://www.goodreads.com/book…",4.34,null,"""goodreads"""
"""1662529376""","""9781662529375""","""Only Santas in the Building""","""https://images-na.ssl-images-a…","""It’s beginning to look a lot l…","""Alexis Daria""",65,"""English""",3047,"""0.99""","""2024-11-12""","""https://www.goodreads.com/book…",3.39,"""Under The Mistletoe Collection""","""goodreads"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1580625339""","""9781580625333""","""Everything Ghost Book""","""https://images-na.ssl-images-a…","""Ghosts and spirits have haunte…","""Jason R. Rich""",289,"""English""",2,null,"""2001-08-01""","""https://www.goodreads.com/book…",3.5,null,"""goodreads"""
"""0879517107""","""9780879517106""","""Everything and More""","""https://images-na.ssl-images-a…","""“Powered by a fast-moving plot…","""Geoff Nicholson""",249,"""English""",5,"""12.99""","""1999-06-01""","""https://www.goodreads.com/book…",3.88,null,"""goodreads"""
"""158062684X""","""9781580626842""","""The Everything Kids' Nature Bo…","""https://images-na.ssl-images-a…","""The natural world holds secret…","""Kathiann M. Kowalski""",144,"""English""",2,"""6.99""","""2002-03-01""","""https://www.goodreads.com/book…",4.0,null,"""goodreads"""


In [87]:
hardcover_book_df = pl.read_csv('datalake/hardcover/hardcover_books.csv', schema={
    "id": pl.String,
    "ISBN_10": pl.String,
    "title": pl.String,
    "description": pl.String,
    "image_url": pl.String,
    "author": pl.String,
    "language": pl.String,
    "publish_date": pl.String,
    "num_pages": pl.Int64,
    "crawl_source": pl.String,
    "hardcover_book_id": pl.Int64
})
print(hardcover_book_df.describe())
hardcover_book_df

shape: (9, 12)
┌────────────┬───────────────┬────────────┬──────────────┬───┬──────────────┬────────────┬──────────────┬──────────────┐
│ statistic  ┆ id            ┆ ISBN_10    ┆ title        ┆ … ┆ publish_date ┆ num_pages  ┆ crawl_source ┆ hardcover_bo │
│ ---        ┆ ---           ┆ ---        ┆ ---          ┆   ┆ ---          ┆ ---        ┆ ---          ┆ ok_id        │
│ str        ┆ str           ┆ str        ┆ str          ┆   ┆ str          ┆ f64        ┆ str          ┆ ---          │
│            ┆               ┆            ┆              ┆   ┆              ┆            ┆              ┆ f64          │
╞════════════╪═══════════════╪════════════╪══════════════╪═══╪══════════════╪════════════╪══════════════╪══════════════╡
│ count      ┆ 223378        ┆ 198761     ┆ 223365       ┆ … ┆ 202630       ┆ 223378.0   ┆ 223378       ┆ 223378.0     │
│ null_count ┆ 0             ┆ 24617      ┆ 13           ┆ … ┆ 20748        ┆ 0.0        ┆ 0            ┆ 0.0          │
│ mean       ┆ nu

id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source,hardcover_book_id
str,str,str,str,str,str,str,str,i64,str,i64
"""9780713469875""","""0713469870""","""Pattern Drafting for Dressmaki…",null,null,"""Pamela Stringer""","""English""","""1993-07-01""",184,"""hardcover""",1189157
"""9781776563531""","""1776563530""","""Head Girl""","""'The first time I read Freya’s…","""https://assets.hardcover.app/e…","""Freya Daly Sadgrove""","""English""","""2020-01-01""",107,"""hardcover""",868944
"""9780811218832""","""081121883X""","""Everything and Nothing""","""A pocket-sized Pearls edition …","""https://assets.hardcover.app/e…","""Jorge Luis Borges""","""English""","""1999-04-01""",129,"""hardcover""",651199
"""9780450054594""","""0450054594""","""Cat's Revenge: More Than 101 U…",null,"""https://assets.hardcover.app/e…","""Philip Lief""","""English""","""1981-01-01""",98,"""hardcover""",1474502
"""9781607051978""","""1607051974""","""Block Party - The Modern Quilt…",null,null,"""Alissa Haight Carlton""",null,"""2011-05-01""",0,"""hardcover""",1189155
…,…,…,…,…,…,…,…,…,…,…
"""9780834842083""","""0834842084""","""Gurdjieff Reconsidered: The Li…",null,null,"""Roger Lipsey""",null,"""2019-02-05""",0,"""hardcover""",1303747
"""9780061288777""",null,"""Celebrate Children's Day/book …","""Children read aloud in various…","""https://assets.hardcover.app/e…","""Pat Mora""","""English""","""2008-01-01""",46,"""hardcover""",6230
"""9781732819214""",null,"""Age of Asango: Book II""",null,"""https://assets.hardcover.app/e…","""Matt Russell""","""English""","""2018-11-16""",425,"""hardcover""",1424537


#### Merging:

In [88]:
columns_to_merge = [
    "id",
    "ISBN_10",
    "title",
    "description",
    "image_url",
    "author",
    "language",
    "publish_date",
    "num_pages",
    "crawl_source"
]

merged_books = pl.concat([
    goodreads_book_df.select(columns_to_merge),
    hardcover_book_df.select(columns_to_merge)   
], how="vertical_relaxed")
merged_books

id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source
str,str,str,str,str,str,str,str,i64,str
"""9781662528392""","""1662528396""","""Cruel Winter with You""","""For two former childhood frien…","""https://images-na.ssl-images-a…","""Ali Hazelwood""","""English""","""2024-11-12""",73,"""goodreads"""
"""9780664266707""","""0664266703""","""The Purpose Gap: Empowering Co…","""In The Purpose Gap, Patrick Re…","""https://images-na.ssl-images-a…","""Patrick B. Reyes""","""English""","""2021-03-16""",256,"""goodreads"""
"""9781662528231""","""166252823X""","""All by My Elf""","""Secret crushes, spicy Christma…","""https://images-na.ssl-images-a…","""Olivia Dade""","""English""","""2024-11-12""",55,"""goodreads"""
"""9781400341122""","""1400341124""","""Mostly What God Does: Reflecti…","""Mostly what God does is love y…","""https://images-na.ssl-images-a…","""Savannah Guthrie""","""English""","""2024-02-20""",302,"""goodreads"""
"""9781662529375""","""1662529376""","""Only Santas in the Building""","""It’s beginning to look a lot l…","""https://images-na.ssl-images-a…","""Alexis Daria""","""English""","""2024-11-12""",65,"""goodreads"""
…,…,…,…,…,…,…,…,…,…
"""9780834842083""","""0834842084""","""Gurdjieff Reconsidered: The Li…",null,null,"""Roger Lipsey""",null,"""2019-02-05""",0,"""hardcover"""
"""9780061288777""",null,"""Celebrate Children's Day/book …","""Children read aloud in various…","""https://assets.hardcover.app/e…","""Pat Mora""","""English""","""2008-01-01""",46,"""hardcover"""
"""9781732819214""",null,"""Age of Asango: Book II""",null,"""https://assets.hardcover.app/e…","""Matt Russell""","""English""","""2018-11-16""",425,"""hardcover"""


In [89]:
merged_books.describe()

statistic,id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source
str,str,str,str,str,str,str,str,str,f64,str
"""count""","""301341""","""276724""","""301328""","""178678""","""266918""","""299969""","""257216""","""279932""",298622.0,"""301341"""
"""null_count""","""0""","""24617""","""13""","""122663""","""34423""","""1372""","""44125""","""21409""",2719.0,"""0"""
"""mean""",null,null,null,null,null,null,null,null,224.973562,null
"""std""",null,null,null,null,null,null,null,null,801.278554,null
"""min""","""0-415-33024-6""","""281573""",""" Captain Underpants and the Bi…","""""","""""",""" Priya Saralkar ""","""Afrikaans""","""+65535-01-01""",0.0,"""goodreads"""
"""25%""",null,null,null,null,null,null,null,null,32.0,null
"""50%""",null,null,null,null,null,null,null,null,208.0,null
"""75%""",null,null,null,null,null,null,null,null,324.0,null
"""max""","""9990100000490""","""None""","""ＤＡＹＳ, Vol. 35""","""🔥 WIRED ROGUE, BOOK 2, ALSO FR…","""https://images-na.ssl-images-a…","""ｉｒｕ""","""Wakashan languages""","""42020-08-04""",380720.0,"""hardcover"""


In [90]:
merged_books.filter(pl.col("id").is_duplicated()).sort("id")

id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source
str,str,str,str,str,str,str,str,i64,str
"""0000199263477""","""0199263477""","""An Introduction to the Philoso…",null,null,"""Brian Davies""",null,"""1982-06-01""",0,"""hardcover"""
"""0000199263477""","""0199263477""","""An Introduction to the Philoso…",null,null,"""Brian Davies""",null,"""1982-06-01""",0,"""hardcover"""
"""0009511176641""",null,"""Tähtilinna""",null,null,"""Tuija Lehtinen""","""English""","""2001-01-01""",302,"""hardcover"""
"""0009511176641""",null,"""Tähtilinna""",null,null,"""Tuija Lehtinen""","""English""","""2001-01-01""",302,"""hardcover"""
"""0009511176641""",null,"""Tähtilinna""",null,null,"""Tuija Lehtinen""","""English""","""2001-01-01""",302,"""hardcover"""
…,…,…,…,…,…,…,…,…,…
"""9847016600173""",null,"""অপেক্ষা""",null,"""https://assets.hardcover.app/e…","""Humayun Ahmed""","""English""","""1970-12-01""",220,"""hardcover"""
"""9847027700831""",null,"""নাটকসমগ্র""",null,"""https://assets.hardcover.app/e…","""Muhammed Zafar Iqbal""","""English""","""2010-02-01""",575,"""hardcover"""
"""9847027700831""",null,"""নাটকসমগ্র""",null,"""https://assets.hardcover.app/e…","""Muhammed Zafar Iqbal""","""English""","""2010-02-01""",575,"""hardcover"""


In [91]:
merged_books_deduplicated = (
    merged_books.sort(by=["id", pl.col("crawl_source") == "hardcover"], descending=True)  # Prioritize "hardcover"
    .group_by("id")
    .agg(
        pl.col("title").first(),                        # Keep the first "title"
        pl.col("crawl_source").first(),  # Keep the first "crawl_source"
        pl.col("ISBN_10").max().fill_null(     pl.col("ISBN_10").min()),  # Fill missing `rating` from `goodreads`
        pl.col("description").max().fill_null( pl.col("description").min()),
        pl.col("image_url").max().fill_null(   pl.col("image_url").min()),
        pl.col("author").max().fill_null(      pl.col("author").min()),
        pl.col("language").first(),
        pl.col("publish_date").max().fill_null(      pl.col("publish_date").min()),
        pl.col("num_pages").max().fill_nan(      pl.col("num_pages").min()),
    )
)

In [92]:
print(merged_books_deduplicated)
merged_books_deduplicated.describe()

shape: (259_154, 10)
┌───────────────┬────────────────┬──────────────┬────────────┬───┬───────────────┬──────────┬──────────────┬───────────┐
│ id            ┆ title          ┆ crawl_source ┆ ISBN_10    ┆ … ┆ author        ┆ language ┆ publish_date ┆ num_pages │
│ ---           ┆ ---            ┆ ---          ┆ ---        ┆   ┆ ---           ┆ ---      ┆ ---          ┆ ---       │
│ str           ┆ str            ┆ str          ┆ str        ┆   ┆ str           ┆ str      ┆ str          ┆ i64       │
╞═══════════════╪════════════════╪══════════════╪════════════╪═══╪═══════════════╪══════════╪══════════════╪═══════════╡
│ 9780835608015 ┆ Sophia:        ┆ goodreads    ┆ 0835608018 ┆ … ┆ Caitlín       ┆ English  ┆ 2001-05-01   ┆ 463       │
│               ┆ Goddess of     ┆              ┆            ┆   ┆ Matthews      ┆          ┆              ┆           │
│               ┆ Wisdom, Bri…   ┆              ┆            ┆   ┆               ┆          ┆              ┆           │
│ 978159143

statistic,id,title,crawl_source,ISBN_10,description,image_url,author,language,publish_date,num_pages
str,str,str,str,str,str,str,str,str,str,f64
"""count""","""259154""","""259144""","""259154""","""238667""","""157482""","""231151""","""258054""","""221719""","""241104""",256452.0
"""null_count""","""0""","""10""","""0""","""20487""","""101672""","""28003""","""1100""","""37435""","""18050""",2702.0
"""mean""",null,null,null,null,null,null,null,null,null,226.960706
"""std""",null,null,null,null,null,null,null,null,null,859.418979
"""min""","""0-415-33024-6""",""" Captain Underpants and the Bi…","""goodreads""","""281573""","""""","""""",""" Priya Saralkar ""","""Afrikaans""","""+65535-01-01""",0.0
"""25%""",null,null,null,null,null,null,null,null,null,36.0
"""50%""",null,null,null,null,null,null,null,null,null,208.0
"""75%""",null,null,null,null,null,null,null,null,null,325.0
"""max""","""9990100000490""","""ＤＡＹＳ, Vol. 35""","""hardcover""","""None""","""🔥 WIRED ROGUE, BOOK 2, ALSO FR…","""https://images-na.ssl-images-a…","""ｉｒｕ""","""Wakashan languages""","""42020-08-04""",380720.0


In [93]:
merged_books_deduplicated.filter(pl.col("id").is_null())

id,title,crawl_source,ISBN_10,description,image_url,author,language,publish_date,num_pages
str,str,str,str,str,str,str,str,str,i64


In [94]:
merged_books_deduplicated.write_csv('datalake/clean_merge/books.csv')

### 2. Genres

In [6]:
hardcover_genre_df = pl.read_csv('datalake/hardcover/hardcover_book_genre.csv', schema={
    "id": pl.String,
    "genre": pl.String,
})
hardcover_genre_df

id,genre
str,str
"""9781776563531""","""Young Adult"""
"""9781776563531""","""Poetry"""
"""9780811218832""","""Classics"""
"""9780811218832""","""Literary Collections"""
"""9780450054594""","""Comics"""
…,…
"""9780307460455""","""Politics"""
"""9781250366788""","""Young Adult"""
"""9781250366788""","""LGBTQ"""


In [12]:
goodreads_genre_df = pl.read_csv('datalake/clean/genres.csv').rename({"name":"genre"})
goodreads_genre_df

genre_id,genre
i64,str
1,"""14th Century"""
2,"""Southern"""
3,"""Agriculture"""
4,"""Australia"""
5,"""Mormonism"""
…,…
1018,"""Basketball"""
1019,"""Witches"""
1020,"""Native American"""


In [29]:
merged_genres = pl.concat([
    goodreads_genre_df.select("genre"),
    hardcover_genre_df.select("genre")
], how="vertical_relaxed")

merged_genres = merged_genres.unique()
merged_genres

genre
str
"""Glück"""
"""Cabala"""
"""Asymmetric warfare"""
"""Children's stories, German"""
"""Justice"""
…
"""Artists' preparatory studies"""
"""Mythology, Roman"""
"""Hindi language"""


In [72]:
merged_genres_stripped = merged_genres.with_columns(
    merged_genres['genre']
    .str.split(",")
    .list.eval(pl.element().str.strip_chars())
    .alias('split_genres')
).explode('split_genres').select('split_genres').unique()
merged_genres_stripped

split_genres
str
"""Amnesia"""
"""Guyana"""
"""Post Colonial"""
"""Cincinnati (Ohio)"""
"""Camino de Santiago de Composte…"
…
"""Stress management"""
"""12th Century"""
"""Qi (Chinese philosophy)"""


In [104]:
merged_genres_cleaned = merged_genres_stripped.with_columns(
    pl.col('split_genres')
    .map_elements(unidecode, return_dtype=pl.String)  # Apply unidecode
    .str.to_lowercase()
    .str.replace_all(r"[&'()+\-./:\[\]?`]", "")
    .str.replace(" ", "-")
    .str.slice(0,15)
    .alias("tag_genres")
)
merged_genres_cleaned

split_genres,tag_genres
str,str
"""Amnesia""","""amnesia"""
"""Guyana""","""guyana"""
"""Post Colonial""","""post-colonial"""
"""Cincinnati (Ohio)""","""cincinnati-ohio"""
"""Camino de Santiago de Composte…","""camino-de santi"""
…,…
"""Stress management""","""stress-manageme"""
"""12th Century""","""12th-century"""
"""Qi (Chinese philosophy)""","""qi-chinese phil"""


In [105]:
unique_chars = set(''.join(merged_genres_cleaned['tag_genres']))
print("Unique Characters:")
print(sorted(unique_chars))

Unique Characters:
[' ', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [5]:
unique_chars = (
    merged_genres_stripped
    .select(pl.col("tag_genres").str.concat(delimiter=""))  # Concatenate all strings
    .unique()  # Get unique characters
)

print(sorted(set(unique_chars.item())))

NameError: name 'merged_genres_stripped' is not defined

### 2. Users

In [119]:
hardcover_user_df = pl.read_csv('datalake/hardcover/hardcover_users.csv').rename({
    'name':'full_name', 'id':'userid_source'
})

# Add source column
hardcover_user_df = hardcover_user_df.with_columns(
    pl.lit('hardcover').alias('account_source')
)

hardcover_user_df

userid_source,full_name,username,books_count,account_source
i64,str,str,i64,str
18783,"""Nitin Rughoonauth""","""Nitin""",7951,"""hardcover"""
22985,"""Court""","""Courtreader""",7474,"""hardcover"""
19573,"""Martha""","""Martha""",6933,"""hardcover"""
13481,"""Deb Nance""","""DebNance""",6773,"""hardcover"""
13318,"""abi""","""booksgamesvinyl""",5666,"""hardcover"""
…,…,…,…,…
16898,"""Yaiza""","""Yai002""",0,"""hardcover"""
14087,"""Stacy ""","""Tacy_0301""",0,"""hardcover"""
2609,"""Thomas Bartensud""","""bartensud""",0,"""hardcover"""


In [120]:
goodreads_user_df = pl.read_csv('datalake/clean/users.csv').rename({
    'name':'full_name','user_id':'userid_source'
})

goodreads_user_df = goodreads_user_df.with_columns(
    pl.lit('goodreads').alias('account_source')
)

goodreads_user_df

userid_source,full_name,profile_url,total_ratings,total_reviews,avg_rating,image_url,account_source
i64,str,str,i64,i64,f64,str,str
19185877,"""acacia""","""https://www.goodreads.com/user…",835,59,4.3,"""https://images.gr-assets.com/u…","""goodreads"""
169105231,"""kelli""","""https://www.goodreads.com/user…",384,1,3.6,"""https://images.gr-assets.com/u…","""goodreads"""
38623456,"""steph""","""https://www.goodreads.com/user…",365,0,3.33,"""https://s.gr-assets.com/assets…","""goodreads"""
8829118,"""mindy""","""https://www.goodreads.com/user…",59,2,3.69,null,"""goodreads"""
179297432,"""kourtney""","""https://www.goodreads.com/user…",64,11,4.28,null,"""goodreads"""
…,…,…,…,…,…,…,…
502678,"""Gail""","""https://www.goodreads.com/user…",0,0,0.0,"""https://i.gr-assets.com/images…","""goodreads"""
31968163,"""The Overflowing Inkwell""","""https://www.goodreads.com/user…",0,0,0.0,"""https://i.gr-assets.com/images…","""goodreads"""
43122073,"""Linda""","""https://www.goodreads.com/user…",0,0,0.0,"""https://i.gr-assets.com/images…","""goodreads"""


In [121]:
goodreads_user_deduplicated = (
    goodreads_user_df.sort(by=["userid_source", "total_ratings"], descending=True)
    .group_by("userid_source")
    .agg(
        pl.col("full_name").first(), 
        pl.col("account_source").first()
    )
)

In [126]:
hardcover_user_deduplicated = (
    hardcover_user_df.sort(by=["userid_source", "books_count"], descending=True)
    .group_by("userid_source")
    .agg(
        pl.col("full_name").first(), 
        pl.col("account_source").first()
    )
)

In [132]:
# Check duplicates
print(goodreads_user_deduplicated.filter(pl.col("userid_source").is_duplicated()).sort("userid_source"))
print(hardcover_user_deduplicated.filter(pl.col("userid_source").is_duplicated()).sort("userid_source"))

shape: (0, 3)
┌───────────────┬───────────┬────────────────┐
│ userid_source ┆ full_name ┆ account_source │
│ ---           ┆ ---       ┆ ---            │
│ i64           ┆ str       ┆ str            │
╞═══════════════╪═══════════╪════════════════╡
└───────────────┴───────────┴────────────────┘
shape: (0, 3)
┌───────────────┬───────────┬────────────────┐
│ userid_source ┆ full_name ┆ account_source │
│ ---           ┆ ---       ┆ ---            │
│ i64           ┆ str       ┆ str            │
╞═══════════════╪═══════════╪════════════════╡
└───────────────┴───────────┴────────────────┘


In [133]:
merged_users = pl.concat([
    goodreads_user_deduplicated.select(["full_name","account_source","userid_source"]),
    hardcover_user_deduplicated.select(["full_name","account_source","userid_source"])   
], how="vertical_relaxed").sort(by=["account_source","userid_source"])

In [207]:
# Add user id (our user id)
merged_users_with_id_cleaned = merged_users.drop_nulls().with_columns(
    pl.arange(1, merged_users.drop_nulls().height+1).alias("new_user_id")  # Generates sequential ids starting from 0
)

In [208]:
print(merged_users_with_id_cleaned)

shape: (40_314, 4)
┌─────────────┬────────────────┬───────────────┬─────────────┐
│ full_name   ┆ account_source ┆ userid_source ┆ new_user_id │
│ ---         ┆ ---            ┆ ---           ┆ ---         │
│ str         ┆ str            ┆ i64           ┆ i64         │
╞═════════════╪════════════════╪═══════════════╪═════════════╡
│ Douglas     ┆ goodreads      ┆ 26            ┆ 1           │
│ Courtney    ┆ goodreads      ┆ 284           ┆ 2           │
│ kaitlin     ┆ goodreads      ┆ 675           ┆ 3           │
│ Scott       ┆ goodreads      ┆ 713           ┆ 4           │
│ erin        ┆ goodreads      ┆ 819           ┆ 5           │
│ …           ┆ …              ┆ …             ┆ …           │
│ Lily Ostler ┆ hardcover      ┆ 23475         ┆ 40310       │
│ Keelin      ┆ hardcover      ┆ 23477         ┆ 40311       │
│ ubyhb       ┆ hardcover      ┆ 23478         ┆ 40312       │
│ Ainsley     ┆ hardcover      ┆ 23481         ┆ 40313       │
│ Jose        ┆ hardcover      ┆ 234

In [209]:
merged_users_with_id_cleaned.write_csv('datalake/clean_merge/users.csv')

### 3. RATINGS

In [201]:
goodreads_rating_df = pl.read_csv('datalake/clean/cleaned_reviews_data.csv',schema={
    "isbn": pl.String,
    "isbn13": pl.String,
    "user_id": pl.Int64,
    "rating_score": pl.Float64,
}).rename({
    'user_id':'userid_source'
})

goodreads_rating_df = goodreads_rating_df.with_columns(
    pl.lit('goodreads').alias('account_source')
)

goodreads_rating_df

isbn,isbn13,userid_source,rating_score,account_source
str,str,i64,f64,str
"""0345465296""","""9780345465290""",1742824,5.0,"""goodreads"""
"""0345465296""","""9780345465290""",9180313,5.0,"""goodreads"""
"""0345465296""","""9780345465290""",36153170,5.0,"""goodreads"""
"""0345465296""","""9780345465290""",22819383,4.0,"""goodreads"""
"""0345465296""","""9780345465290""",8845508,5.0,"""goodreads"""
…,…,…,…,…
"""0399155341""","""9780399155345""",125982174,5.0,"""goodreads"""
"""0439554934""","""9780439554930""",125982174,5.0,"""goodreads"""
"""0316015849""","""9780316015844""",125982174,5.0,"""goodreads"""


In [213]:
# Merge to get new id
goodreads_rating_df_good = goodreads_rating_df.join(
    merged_users_with_id_cleaned, on=["userid_source","account_source"], how="left"
).drop_nulls("new_user_id")

goodreads_rating_df_good

isbn,isbn13,userid_source,rating_score,account_source,full_name,new_user_id
str,str,i64,f64,str,str,i64
"""0345465296""","""9780345465290""",1742824,5.0,"""goodreads""","""Jason Koivu""",3145
"""0345465296""","""9780345465290""",22819383,4.0,"""goodreads""","""Timothy Boyd""",11776
"""0345465296""","""9780345465290""",929238,4.0,"""goodreads""","""H (no longer expecting notific…",1809
"""0345465296""","""9780345465290""",57773209,3.0,"""goodreads""","""Chris Seltzer""",15652
"""0815602057""","""9780815602057""",316102,3.0,"""goodreads""","""Krista the Krazy Kataloguer""",863
…,…,…,…,…,…,…
"""0399155341""","""9780399155345""",125982174,5.0,"""goodreads""","""lacey""",19640
"""0439554934""","""9780439554930""",125982174,5.0,"""goodreads""","""lacey""",19640
"""0316015849""","""9780316015844""",125982174,5.0,"""goodreads""","""lacey""",19640


In [214]:
goodreads_rating_df_good.filter(pl.col('full_name').is_null())

isbn,isbn13,userid_source,rating_score,account_source,full_name,new_user_id
str,str,i64,f64,str,str,i64


In [222]:
hardcover_rating_df = pl.read_csv('datalake/hardcover/hardcover_ratings.csv',schema={
    "rating": pl.Float64,
    "book_id": pl.Int64,
    "user_id": pl.Int64,
    "date_added": pl.String,
    "edition_id": pl.Int64
}).rename({
    'book_id': 'hardcover_book_id',
    'user_id':'userid_source',
    'rating':'rating_score'
})

hardcover_rating_df = hardcover_rating_df.with_columns(
    pl.lit('hardcover').alias('account_source')
)

print(hardcover_rating_df.describe())
hardcover_rating_df

shape: (9, 7)
┌────────────┬──────────────┬───────────────────┬───────────────┬────────────┬─────────────┬────────────────┐
│ statistic  ┆ rating_score ┆ hardcover_book_id ┆ userid_source ┆ date_added ┆ edition_id  ┆ account_source │
│ ---        ┆ ---          ┆ ---               ┆ ---           ┆ ---        ┆ ---         ┆ ---            │
│ str        ┆ f64          ┆ f64               ┆ f64           ┆ str        ┆ f64         ┆ str            │
╞════════════╪══════════════╪═══════════════════╪═══════════════╪════════════╪═════════════╪════════════════╡
│ count      ┆ 1.631332e6   ┆ 1.631332e6        ┆ 1.631332e6    ┆ 1631332    ┆ 1.628765e6  ┆ 1631332        │
│ null_count ┆ 0.0          ┆ 0.0               ┆ 0.0           ┆ 0          ┆ 2567.0      ┆ 0              │
│ mean       ┆ 3.854516     ┆ 543281.600496     ┆ 10790.778025  ┆ null       ┆ 2.4687e7    ┆ null           │
│ std        ┆ 0.96229      ┆ 378920.3298       ┆ 6541.648061   ┆ null       ┆ 9.4193e6    ┆ null         

rating_score,hardcover_book_id,userid_source,date_added,edition_id,account_source
f64,i64,i64,str,i64,str
3.0,1111086,6359,"""0007-07-17""",31135094,"""hardcover"""
5.0,265718,20299,"""0008-01-18""",3537983,"""hardcover"""
4.0,378793,20299,"""0008-02-14""",8631567,"""hardcover"""
5.0,427391,20299,"""0008-02-14""",30193579,"""hardcover"""
4.0,429541,20299,"""0008-02-14""",19508505,"""hardcover"""
…,…,…,…,…,…
3.0,501689,11761,"""2021-04-19""",30451470,"""hardcover"""
3.0,437921,10357,"""2021-04-19""",31263374,"""hardcover"""
5.0,1238232,10357,"""2021-04-19""",31263253,"""hardcover"""


In [223]:
# Merge to get new id
hardcover_rating_df_with_user = hardcover_rating_df.join(
    merged_users_with_id_cleaned, on=["userid_source","account_source"], how="left"
).drop_nulls("new_user_id")

hardcover_rating_df_with_user

rating_score,hardcover_book_id,userid_source,date_added,edition_id,account_source,full_name,new_user_id
f64,i64,i64,str,i64,str,str,i64
3.0,1111086,6359,"""0007-07-17""",31135094,"""hardcover""","""Shannon""",27922
5.0,265718,20299,"""0008-01-18""",3537983,"""hardcover""","""SSK""",38008
4.0,378793,20299,"""0008-02-14""",8631567,"""hardcover""","""SSK""",38008
5.0,427391,20299,"""0008-02-14""",30193579,"""hardcover""","""SSK""",38008
4.0,429541,20299,"""0008-02-14""",19508505,"""hardcover""","""SSK""",38008
…,…,…,…,…,…,…,…
3.0,501689,11761,"""2021-04-19""",30451470,"""hardcover""","""j""",31919
3.0,437921,10357,"""2021-04-19""",31263374,"""hardcover""","""Andrei""",30887
5.0,1238232,10357,"""2021-04-19""",31263253,"""hardcover""","""Andrei""",30887


In [228]:
hardcover_rating_df_with_user.sort(by=["hardcover_book_id","new_user_id"])

rating_score,hardcover_book_id,userid_source,date_added,edition_id,account_source,full_name,new_user_id
f64,i64,i64,str,i64,str,str,i64
3.0,1,641,"""2021-01-21""",759160,"""hardcover""","""winters""",23874
4.0,1,5870,"""2014-04-25""",25224958,"""hardcover""","""Spriggana""",27546
5.0,1,7178,"""2024-09-16""",25224958,"""hardcover""","""Esme""",28511
3.0,1,10282,"""2022-06-25""",25224958,"""hardcover""","""Anki""",30823
4.0,1,12795,"""2015-06-25""",759160,"""hardcover""","""Laura""",32671
…,…,…,…,…,…,…,…
5.0,1630673,23250,"""2014-03-13""",31709666,"""hardcover""","""Daria""",40145
2.0,1630675,23250,"""2024-04-26""",31709663,"""hardcover""","""Daria""",40145
3.0,1630676,23250,"""2024-04-26""",31709667,"""hardcover""","""Daria""",40145


In [232]:
# Get isbn
hardcover_rating_df_good = hardcover_rating_df_with_user.join(
    hardcover_book_df, on="hardcover_book_id", how="left"
    
).drop_nulls("id").select(
    ["rating_score", "new_user_id","account_source","id"]
).rename({'id':'isbn13'})

hardcover_rating_df_good

rating_score,hardcover_book_id,userid_source,date_added,edition_id,account_source,full_name,new_user_id,id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source
f64,i64,i64,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str
4.0,189532,20299,"""0008-02-25""",16650573,"""hardcover""","""SSK""",38008,"""9780345486455""","""0345486455""","""Healthy Sleep Habits, Happy Ch…","""One of the country's leading r…","""https://assets.hardcover.app/e…","""Marc Weissbluth""","""English""","""1987-01-01""",542,"""hardcover"""
4.0,189532,20299,"""0008-02-25""",16650573,"""hardcover""","""SSK""",38008,"""9780345486455""","""0345486455""","""Healthy Sleep Habits, Happy Ch…","""One of the country's leading r…","""https://assets.hardcover.app/e…","""Marc Weissbluth""","""English""","""1987-01-01""",542,"""hardcover"""
4.0,189532,20299,"""0008-02-25""",16650573,"""hardcover""","""SSK""",38008,"""9780345486455""","""0345486455""","""Healthy Sleep Habits, Happy Ch…","""One of the country's leading r…","""https://assets.hardcover.app/e…","""Marc Weissbluth""","""English""","""1987-01-01""",542,"""hardcover"""
4.0,331156,20299,"""0008-03-13""",26657225,"""hardcover""","""SSK""",38008,"""9782378981532""","""2378981538""","""Think and Grow Rich""","""You are currently reading one …","""https://assets.hardcover.app/e…","""Napoleon Hill""","""English""","""1937-01-01""",128,"""hardcover"""
4.0,331156,20299,"""0008-03-13""",26657225,"""hardcover""","""SSK""",38008,"""9782378981532""","""2378981538""","""Think and Grow Rich""","""You are currently reading one …","""https://assets.hardcover.app/e…","""Napoleon Hill""","""English""","""1937-01-01""",128,"""hardcover"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2.0,1474278,17635,"""2021-04-19""",31520385,"""hardcover""","""PB""",36071,"""9780722158753""","""0722158750""","""Baal""","""A woman is raped and a child i…","""https://assets.hardcover.app/e…","""Robert R. McCammon""","""English""","""1978-08-01""",251,"""hardcover"""
5.0,432523,17635,"""2021-04-19""",30401505,"""hardcover""","""PB""",36071,"""9780789156327""","""0789156326""","""A Sound of Thunder""",null,"""https://assets.hardcover.app/e…","""Ray Bradbury""","""English""","""1951-01-01""",41,"""hardcover"""
4.0,259662,2488,"""2021-04-19""",2710952,"""hardcover""","""Matt Woolery """,25138,"""9780770430078""","""0770430074""","""Life of Pi""","""After the tragic sinking of a …","""https://assets.hardcover.app/e…","""Yann Martel""","""English""","""1980-01-01""",480,"""hardcover"""


In [225]:
columns_to_get = ["rating_score","new_user_id","isbn13","account_source"]

merged_rating = pl.concat([
    goodreads_rating_df_good.select(columns_to_get),
    hardcover_rating_df_good.select(columns_to_get)   
], how="vertical_relaxed")

In [226]:
merged_rating

rating_score,new_user_id,isbn13,account_source
f64,i64,str,str
5.0,3145,"""9780345465290""","""goodreads"""
4.0,11776,"""9780345465290""","""goodreads"""
4.0,1809,"""9780345465290""","""goodreads"""
3.0,15652,"""9780345465290""","""goodreads"""
3.0,863,"""9780815602057""","""goodreads"""
…,…,…,…
2.0,36071,"""9780722158753""","""hardcover"""
5.0,36071,"""9780789156327""","""hardcover"""
4.0,25138,"""9780770430078""","""hardcover"""


In [220]:
merged_rating.write_csv('datalake/clean_merge/ratings.csv')